# Lung field segmentation: leave-one-dataset-out runs

Trains three arms on three folds, nine runs, and writes one per-image CSV each.

**Use "Save & Run All (Commit)", not an interactive session.** Output written
interactively is deleted when the session closes, so an interactive run that
finishes and is then closed leaves nothing behind.

**Set the accelerator to T4 x2.** `torch.cuda.is_available()` returns True on a
P100 that cannot launch a kernel, because Kaggle's torch is built for sm_70+ and
the P100 is sm_60. The preflight cell below launches a real kernel instead of
trusting the flag.

Attach both datasets before running: `nikhilpandey360/chest-xray-masks-and-labels`
and `abduzzami/jsrt-247-image-lung-segmentation-mask-dataset`.

Nine runs may not fit in one session. To continue in a second session, attach the
previous session's output through **Add Data > Your Work > Notebook Output** and
put its mount path in `PREVIOUS_OUTPUT` below. Every session starts with an empty
`/kaggle/working`, so without that step the resume check finds nothing and the
matrix restarts from the first run every time.

In [ ]:
REPO_URL = "https://github.com/Tayyab885/chest-xray-lung-segmentation.git"
SEED = 42

# Mount paths of earlier sessions' output, in the order they ran. Empty on the
# first session. Each entry is the directory that holds results/per_image, so
# "/kaggle/input/<notebook-slug>/results" rather than the mount root.
PREVIOUS_OUTPUT = []

# Kaggle kills a committed session at twelve hours and keeps nothing from a
# session it killed, including runs that had already finished. Stopping early
# and exiting cleanly is what preserves them.
BUDGET_HOURS = 10.5

assert REPO_URL, "set REPO_URL to the clone URL before running"

In [ ]:
import os, shutil, subprocess

# Cloned to /tmp rather than /kaggle/working so the committed output holds only
# checkpoints and results, not a copy of the source.
REPO = "/tmp/repo"
if os.path.exists(REPO):
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
os.chdir(REPO)
print(subprocess.run(["git", "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout)

In [ ]:
import torch
from src.train import assert_gpu_usable

device = assert_gpu_usable()
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "")
assert device == "cuda", "no usable GPU: check the accelerator is T4 x2, not P100"

# Only cuda:0 is used and there is no mixed precision. Both are wall clock, not
# correctness: the second T4 sits idle and the runs take longer than the
# hardware allows.
print("visible GPUs:", torch.cuda.device_count())

In [ ]:
# The globs in configs/kaggle_layout.yaml were inferred from the dataset's file
# listing, not from a local copy. Print the real tree before trusting them.
!ls /kaggle/input
!find /kaggle/input -maxdepth 3 -type d | head -40

In [ ]:
import yaml
from src.config import load_config
from src.data import build_manifest
from src.splits import lodo_folds

cfg = load_config("configs/kaggle.yaml")
cfg["seed"] = SEED
layout = yaml.safe_load(open("configs/kaggle_layout.yaml", encoding="utf-8"))

# This is the check on the globs above. build_manifest counts every source
# against expected_count and raises on a mismatch or a missing mask, so a wrong
# path costs this cell rather than a training run. If it raises, fix
# configs/kaggle_layout.yaml from the tree printed above and commit the fix.
manifest = build_manifest(cfg["data_root"], layout)
print(manifest.groupby("source").size())

folds = lodo_folds(manifest, cfg["val_frac"], cfg["test_frac"], cfg["seed"])
for fold in folds:
    print(fold.held_out, "| train", len(fold.train), "val", len(fold.val),
          "| in-domain test", len(fold.id_test), "| off-domain test", len(fold.ood_test))

In [ ]:
import time
from src.run_matrix import restore_previous, run_all

# Must come before run_all: it is what lets the resume check see the runs an
# earlier session already scored.
restore_previous(cfg, PREVIOUS_OUTPUT)

result = run_all(cfg, folds, deadline=time.monotonic() + BUDGET_HOURS * 3600)

print("\nwritten this session:", len(result["written"]))
print("skipped (already scored):", len(result["skipped"]))
print("failed:", [run for run, _ in result["failed"]])
print("remaining, attach this session's output and run again:", result["remaining"])

In [ ]:
# What the commit will carry back. Tables and figures are generated locally by
# src/run_report.py, not here, so a change to the reporting code does not need
# another GPU session.
#
# per_image holds every run restored plus every run scored here, but the
# checkpoint directory holds only this session's. The overlay figures need all
# nine, so pull each session's output and merge them locally.
!ls -la /kaggle/working/results/per_image
!ls -la /kaggle/working/checkpoints